# **正则化**

- L1正则化：在损失函数中额外加入模型所有参数的绝对值之和作为惩罚项。
  - 训练时模型会尽可能让参数变小来降低loss值
  - $Loss_{L1}=Loss(\theta) + \lambda \sum_i|\theta_i|$
  - 如果数据量越少，模型越复杂，输入特征越多，那么$\lambda$设置越大

- L2正则化：在损失函数中加入模型参数的平方和作为惩罚项。
  - $Loss_{L2}=Loss(\theta) + \lambda \sum_i\theta_i^2$

- 区别：L1正则化会让模型很多参数为0，也就是模型参数会变得稀疏；L2正则化会让模型参数变小，但是不会等于0
  - 因为L1正则化是以绝对值，而根据绝对值函数的导数知道它是以固定的速度降低参数的绝对值直到为0；L2正则化是以平方项，导数是$2x$，当$x$越接近于0时梯度越小速度越慢，所以参数只会接近0而不会等于0

在pytorch中加入L2正则化
```
l2_norm = 0.0
for param in model.parameters():
    l2_norm += param.pow(2).sum()   # 遍历所有参数，进行平方求和
loss = criterion(outputs, labels) + 1e-4 * l2_norm
```

# 动量梯度下降 Momentum Gradient Descent

- 梯度下降的问题：
    - 接近最优点时产生震荡（梯度正负交替出现），永远到不了最优点；
    - 当某一参数有局部最优时，即loss对该参数的偏导数在某个很小的局部接近0，由于每一步只看当前梯度（较小），此时参数更新非常慢，停滞不前

- 动量梯度下降：指数加权平均的方法来更新参数。**抑制震荡，惯性加速**
    - 训练震荡：梯度正负交替，经过加权平均后正负相互抵消；
    - 局部最优：考虑历史值，不会因为当前梯度值为0就停滞，而是依旧沿着历史方向继续更新

动量梯度的更新过程：以参数$w$为例，

- 计算$w$的梯度：$g_w = \frac{\partial loss}{\partial w}$
- 定义变量$V_w$表示$g_w$的指数加权平均值，更新自身值：$V_w = \beta V_w + (1-\beta)g_w$
- 更新参数$w$：$w = w - lr \cdot V_w$

\* 训练过程中，一直要保存一个指数加权平均值$V_w$

---

在pytorch里面实现：
```
optimizer = torch.optim.SGD(model.parameters(), lr=学习率, momentum=动量系数)
```

# RMSProp(Root Mean Square Propagation optimizer) 均方根传播优化器

- 统一学习率：所有参数都用同一个学习率。梯度大的参数容易震荡，梯度小的参数不收敛；
- 自适应学习率：让参数更新的梯度值都除以一个数。记录每个参数梯度平方的指数加权平均值$S$，更新参数时的梯度除以$\sqrt{S}$。从而实现减小大梯度参数的学习率，增大小梯度参数的学习率

---

RMSProp的计算过程：以参数$w$为例，

- 计算$w$的梯度：$g_w = \frac{\partial loss}{\partial w}$
- 计算${g_w}^2$的指数加权平均值：$S_w = \beta S_w + (1-\beta){g_w}^2$
- 更新参数$w$：$w = w - \frac{lr}{\sqrt{S_w}+\epsilon} \cdot g_w$

\* 其中，衰减系数$\beta$一般取0.9，$\epsilon$为了防止除0一般是1e-8

---

在pytorch里实现：
```
optimizer = torch.optim.RMSprop(model.parameters(), lr=学习率, alpha=衰减系数, eps=1e-8)
```

# Adam优化器 Adaptive Moment Estimation

同时利用动量给梯度更新增加惯性和震荡阻尼，也利用历史数据的均方根来自适应调整学习率

---

Adam的计算过程：以参数$w$为例，

- 计算$w$的梯度：$g_w = \frac{\partial loss}{\partial w}$
- 计算并更新一阶矩指数加权平均值$V_w$：$V_w = \beta_1 V_w + (1-\beta_1)V_w$
- 和二阶矩指数加权平均值$S_w$：$S_w = \beta_2 S_w + (1-\beta_2){g_w}^2$
- 校正：$V_w^{correct} = \frac{V_w}{1-\beta^t_1}$, $S_w^{correct} = \frac{S_w}{1-\beta^t_2}$
- 更新参数$w$：$w = w - lr \cdot \frac{V_w^{correct}}{\sqrt{S_w^{correct}} + \epsilon}$

\* 其中，一般$\beta_1=0.9, \beta_2=0.999$，训练时每个参数都需要额外保存 V 和 S 两个值

---

在pytorch里实现Adam：
```
optimizer = torch.optim.adma(model.parameters(), lr=学习率, betas=(beta1, beta2))
```

# 权重衰减 Weight Decay

- 防止过拟合。在训练的每一步更新参数时，同时缩小参数值，防止参数绝对值过大。
- 与正则化不同，虽然都是减小参数的绝对值，但正则是在loss函数里面增加额外项，而权重衰减是直接减小参数的绝对值
- $w_t = w_t - lr \times g_w - lr \times \lambda w_t$

\* 其中，$\lambda$是权重衰减系数，一般在1e-2到1e-4，过拟合严重就要设置更大

---

在pytorch里面实现权重衰减：
```
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, weight_decay=1e-4)
```

# Dropout

- 训练时在隐藏层中随机禁用一些神经元。从简化网络结构的方面解决过拟合
- 设置概率 p，每个神经元都有p的概率被禁用，对于保留的神经元的输出需要除以 1-p 进行缩放调整，确保进入下一层的期望不变
    - 保证训练分布约等于推理分布
    - 不缩放的话多层之后期望为$(1-p)^L \to 0$，激活几乎消失，梯度变小，难以训练
    - p 一般取0.1-0.5，p越大正则化效果越强

---

pytorch里面实现dropout：
```
class NeuralNetwork(nn.Module):
    def __init__(self):
        super.__init__()
        self.model = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.model(x)
```

# 批量归一化 Batch Normalization

- 之前对输入数据进行归一化，让训练更稳定快速。但随着网络的前向传播，深处层的输入取值范围又会被不断拉大；
- 一般情况下，是对激活前的z值进行归一化；
- 如果一个线性层后边要加批量归一化，那么这一层就可以不设置偏置项（因为无论学到的偏置b是多少，减去均值后结果都是一样的）。另外输出层一般不加批量归一化层；
- 批量归一化可以保证每一层的输出的分布保持稳定的均值和方差，深层网络可以在这个稳固的基础上进行抽象特征的学习；
- 防止过拟合。批量归一化过程中每次计算的均值和标准差都不一样，数据经过变换后，相当于引入了噪声；
- 训练和推理不同：训练时，BatchNorm使用当前batch的z值均值和标准差进行归一化，并用指数加权平均的方法更新和维护z的均值和标准差；推理时，使用指数加权平均得到的均值和标准差来归一化推理时的z值

---

pytorch里实现BatchNorm：
```
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(28 * 28, 128, bias=False),
            nn.BatchNorm1d(128),                    # nn.BatchNorm1d(num_features)
            nn.ReLU(),

            nn.Linear(128, 128, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, 128, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, 64, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.model(x)
```